# 01 · Candidate Sourcing — NPPES NPI Registry
**Source:** CMS NPPES API v2.1.

**Strategy:** query every Manhattan ZIP (10001–10282) by *practice location*, instead of `city=NEW YORK`,
to avoid the 1,200-result cap and inconsistent city spellings.

In [19]:
import sys; sys.path.append("..")
import pandas as pd
from src import config, nppes

REFRESH = config.REFRESH

### Gets the raw NPPES data from the API or from saved cache

In [20]:
raw = nppes.load_or_fetch(refresh=REFRESH)
print(f"Fetched at {raw['fetched_at']} | {len(raw['results'])} unique NPIs | "
      f"{sum(1 for c in raw['per_zip_counts'].values() if c)} ZIPs with dermatology records")

Fetched at 2026-11-15T21:49:07.861880+00:00 | 980 unique NPIs | 39 ZIPs with dermatology records


### Exploring the raw data before filtering

- **NPI type** (`enumeration_type`): `NPI-1` = individual provider, `NPI-2` = organization
  ([NPPES API docs](https://npiregistry.cms.hhs.gov/api-page))
- **Specialty** (`taxonomies`): NUCC Health Care Provider Taxonomy codes, e.g. `207N00000X` = Dermatology
  ([NUCC code set](https://taxonomy.nucc.org/))


In [21]:
import json
sample = next(r for r in raw["results"] if len(r.get("taxonomies", [])) > 1)
print("=== What one raw NPPES record looks like ===\n")
print(json.dumps({"number": sample["number"],
                  "enumeration_type": sample["enumeration_type"],
                  "taxonomies": [{k: t.get(k) for k in ("code", "desc", "primary")} for t in sample["taxonomies"]]},
                 indent=2))

=== What one raw NPPES record looks like ===

{
  "number": "1689627184",
  "enumeration_type": "NPI-2",
  "taxonomies": [
    {
      "code": "207N00000X",
      "desc": "Dermatology",
      "primary": false
    },
    {
      "code": "207R00000X",
      "desc": "Internal Medicine",
      "primary": false
    },
    {
      "code": "207RH0003X",
      "desc": "Internal Medicine, Hematology & Oncology",
      "primary": false
    },
    {
      "code": "207RI0011X",
      "desc": "Internal Medicine, Interventional Cardiology",
      "primary": false
    },
    {
      "code": "207RN0300X",
      "desc": "Internal Medicine, Nephrology",
      "primary": false
    },
    {
      "code": "207T00000X",
      "desc": "Neurological Surgery",
      "primary": false
    },
    {
      "code": "207V00000X",
      "desc": "Obstetrics & Gynecology",
      "primary": false
    },
    {
      "code": "207W00000X",
      "desc": "Ophthalmology",
      "primary": false
    },
    {
      "code": "207

In [25]:
print("=== NPI types in the data ===\n")

NPI_TYPES = {"NPI-1": "Individual provider (one person, e.g. a dermatologist)",
             "NPI-2": "Organization (practice, group, or hospital entity)"}
(df.enum_type.value_counts().rename_axis("enum_type").to_frame("records")
   .assign(meaning=lambda t: t.index.map(NPI_TYPES)))

=== NPI types in the data ===



,records,meaning
enum_type,,
NPI-1,701,"Individual provider (one person, e.g. a dermat..."
NPI-2,279,"Organization (practice, group, or hospital ent..."


In [12]:
print("=== Every dermatology code that appeared, and how often ===\n")
tax = pd.DataFrame([{"npi": r["number"], "enum_type": r["enumeration_type"],
                     "code": t.get("code"), "desc": t.get("desc") or "", "primary": bool(t.get("primary"))}
                    for r in raw["results"] for t in r.get("taxonomies", [])])

derm_codes = (tax[tax.desc.str.startswith("Dermatology")]
              .groupby(["code", "desc"])
              .agg(npis=("npi", "nunique"), as_primary=("primary", "sum"))
              .sort_values("npis", ascending=False))
derm_codes


=== Every dermatology code that appeared, and how often ===



,,npis,as_primary
code,desc,,
207N00000X,Dermatology,878,784
207NS0135X,"Dermatology, Procedural Dermatology",95,54
207ND0101X,"Dermatology, MOHS-Micrographic Surgery",92,58
207ND0900X,"Dermatology, Dermatopathology",46,18
207NP0225X,"Dermatology, Pediatric Dermatology",25,11
207NI0002X,"Dermatology, Clinical & Laboratory Dermatological Immunology",13,4


### Filters and funnel table

- Manhattan ZIP: keeps rows whose practice-address ZIP starts with 100, 101 or 102.
- Primary taxonomy = Dermatology: keeps rows whose main specialty starts with "Dermatology"
- Exclude dermatopathology: removes dermatologists whose main specialty is code 207ND0900X

In [14]:
DERMPATH = "207ND0900X"
funnel = []

def log(label, d):
    funnel.append({"step": label, "NPI-1": (d.enum_type == "NPI-1").sum(),
                   "NPI-2": (d.enum_type == "NPI-2").sum(), "total": len(d)})
    return d

d = log("Raw API results", df)
d = log("Practice location in Manhattan ZIP", d[d.zip5.str.startswith(config.MANHATTAN_ZIP_PREFIXES)])
d = log("Primary taxonomy = Dermatology", d[d.taxonomy_desc.fillna("").str.startswith("Dermatology")])
d = log("Exclude dermatopathology (lab-only)", d[d.taxonomy_code != DERMPATH])

providers = d.reset_index(drop=True)
pd.DataFrame(funnel)

,step,NPI-1,NPI-2,total
0,Raw API results,701,279,980
1,Practice location in Manhattan ZIP,606,274,880
2,Primary taxonomy = Dermatology,597,235,832
3,Exclude dermatopathology (lab-only),588,231,819


### Sanity checks

In [16]:
display(providers.zip5.value_counts().head(10).to_frame("providers"))
display(providers.taxonomy_desc.value_counts().to_frame("providers"))
display(providers[providers.enum_type == "NPI-2"]
        [["name", "dba", "street_raw", "zip5", "auth_official_first", "auth_official_last", "auth_official_title"]].head(10))
print("Orgs with parent_org:", providers.parent_org.notna().sum(),
      "| with DBA:", providers.dba.notna().sum(),
      "| NPPES record >5 yrs old:", (providers.years_since_update > 5).sum())

,providers
zip5,
10016,107
10021,100
10022,66
10065,55
10003,47
10029,47
10028,45
10012,39
10019,39


,providers
taxonomy_desc,
Dermatology,712
"Dermatology, Procedural Dermatology",51
"Dermatology, MOHS-Micrographic Surgery",44
"Dermatology, Pediatric Dermatology",9
"Dermatology, Clinical & Laboratory Dermatological Immunology",3


,name,dba,street_raw,zip5,auth_official_first,auth_official_last,auth_official_title
0,CARLOS A CHARLES MD PC,DERMA DI COLORE,115 W 27TH ST SUITE 1001,10001,CARLOS,CHARLES,Owner
2,"HIGH LINE DERMATOLOGY SERVICES, P.C.",NaN,514 W 28TH ST,10001,HANA,JEON,MD
9,BETH ISRAEL MEDICAL CENTER,BIMC DERMATOLOGY CONSULTANTS,10 UNION SQ E SUTIE 3C,10003,VINCENT,DELEO,DEPARTMENT CHAIR
10,BIMC FACULTY PRACTICE,NaN,10 UNION SQ E SUITE 3C,10003,DEBORAH,HACKETT,VP
13,COSMETIC SURGERY AND DERMATOLOGY PLLC,NaN,117 E 18TH STREET SUITE 1,10003,OMAR,TORRES,PRESIDENT
14,"DAVID A. COLBERT M.D., P.C.",NEW YORK DERMATOLOGY GROUP,119 5TH AVE 4TH FLOOR,10003,NANCY,FONSECA,Claims Manager
19,GRACE DERMATOLOGY PC,NaN,51 5TH AVE SUITE A1,10003,HYEWON,PAK,Physician
23,ICAHN SCHOOL OF MEDICINE AT MOUNT SINAI,DERMATOLOGY DEPARTMENT OF MOUNT SINAI BISLR,10 UNION SQUARE EAST,10003,CRYSTAL,MACNEILL,"Vice President, CBO Director"
24,INTERGRAL MEDICAL HEALTH WELLNESS PLLC,INTERGRAL MEDICAL HEALTH WELLNESS PLLC,119 5TH AVE FL 2,10003,DAVID,COLBERT,CEO
25,"JEFFREY AINSPAN M.D., P.C.",NaN,308A E 15TH ST,10003,ELLIE,AINSPAN,Office Manager


Orgs with parent_org: 0 | with DBA: 47 | NPPES record >5 yrs old: 484


In [17]:
out = config.INTERIM / "providers_flat.parquet"
providers.to_parquet(out, index=False)
print(f"Saved {len(providers)} rows → {out.relative_to(config.ROOT)}")

Saved 819 rows → data/interim/providers_flat.parquet


### Findings
- 980 unique NPIs (701 individuals, 279 organizations).
- ~100 records had their primary practice outside Manhattan → removed.
- Dermatopathologists excluded: lab-based, no longitudinal patient tracking.
- API returns only active NPIs; status filter kept as a safeguard.
- 484 records haven't been updated in NPPES in 5+ years